In [ ]:
#-----WORKING--------TRAIN-------------------
# 🔷 1. Upload dataset
from google.colab import files
uploaded = files.upload()

# 🔷 2. Imports
import pandas as pd
import io
from sklearn.tree import DecisionTreeClassifier, export_text

# 🔷 3. Load dataset
file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))
else:
    df = pd.read_excel(io.BytesIO(uploaded[file_name]))

print("Dataset Loaded ✅")

# 🔷 4. Clean column names
df.columns = df.columns.str.strip()

# 🔷 🔥 5. Convert LOW/MID/HIGH → numbers (if present)
mapping = {'LOW': 0, 'MID': 1, 'HIGH': 2}

for col in ['Temperature num', 'Rainfall num', 'Humidity num', 'Yield']:
    if df[col].dtype == 'object':
        df[col] = df[col].map(mapping)

# 🔷 6. Ensure all values are numeric
df = df.dropna()   # remove any invalid rows

# 🔷 🔶 7. Define FEATURES

# Crop Model Inputs
X_crop = df[[
    'Temperature num',
    'Rainfall num',
    'Humidity num',
    'Soil num',
    'Weather num'
]]
y_crop = df['Crop']

# Yield Model Inputs (🔥 includes Crop)
X_yield = df[[
    'Temperature num',
    'Rainfall num',
    'Humidity num',
    'Soil num',
    'Weather num',
    'Crop'
]]
y_yield = df['Yield']

# 🔷 🔥 8. Train models (optimized)

model_crop = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

model_yield = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)

model_crop.fit(X_crop, y_crop)
model_yield.fit(X_yield, y_yield)

print("\nModels Trained Successfully ✅")

# 🔷 🔥 9. Extract Decision Rules (FPGA use)
print("\n--- CROP DECISION RULES ---")
print(export_text(model_crop, feature_names=list(X_crop.columns)))

print("\n--- YIELD DECISION RULES ---")
print(export_text(model_yield, feature_names=list(X_yield.columns)))

Saving train_data.csv to train_data.csv
Dataset Loaded ✅

Models Trained Successfully ✅

--- CROP DECISION RULES ---
|--- Weather num <= 2.50
|   |--- Rainfall num <= 1.50
|   |   |--- Soil num <= 0.50
|   |   |   |--- Humidity num <= 1.50
|   |   |   |   |--- Weather num <= 1.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- Weather num >  1.50
|   |   |   |   |   |--- class: 2
|   |   |   |--- Humidity num >  1.50
|   |   |   |   |--- Weather num <= 1.50
|   |   |   |   |   |--- class: 4
|   |   |   |   |--- Weather num >  1.50
|   |   |   |   |   |--- class: 3
|   |   |--- Soil num >  0.50
|   |   |   |--- Soil num <= 2.50
|   |   |   |   |--- Temperature num <= 1.50
|   |   |   |   |   |--- class: 3
|   |   |   |   |--- Temperature num >  1.50
|   |   |   |   |   |--- class: 1
|   |   |   |--- Soil num >  2.50
|   |   |   |   |--- Weather num <= 1.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- Weather num >  1.50
|   |   |   |   |   |--- class: 1
|   |--- Rainfal

In [ ]:
# ------------------ FINAL TEST CODE ------------------

# 🔷 1. USER INPUT (WITH THRESHOLDS)

print("Enter values (0=LOW, 1=MID, 2=HIGH)\n")

print("Temperature: 0(<20°C), 1(20–30°C), 2(>30°C)")
temp = int(input("Temperature (0/1/2): "))

print("\nRainfall: 0(<300 mm), 1(300–700 mm), 2(>700 mm)")
rain = int(input("Rainfall (0/1/2): "))

print("\nHumidity: 0(<50%), 1(50–75%), 2(>75%)")
humidity = int(input("Humidity (0/1/2): "))

print("\nWeather: 0=Sunny, 1=Rainy, 2=Cloudy, 3=Stormy")
weather = int(input("Weather: "))

print("\nSoil: 0=Sandy, 1=Loamy, 2=Clay, 3=Peaty, 4=Silty")
soil = int(input("Soil: "))


# 🔷 2. CREATE INPUT DATA
import pandas as pd

base_input = pd.DataFrame([[
    temp, rain, humidity, soil, weather
]], columns=[
    'Temperature num',
    'Rainfall num',
    'Humidity num',
    'Soil num',
    'Weather num'
])


# 🔷 🔥 3. MAPPINGS

crop_map = {
    0: "Rice",
    1: "Wheat",
    2: "Corn",
    3: "Barley",
    4: "Soybeans"
}

yield_level_map = {
    0: "LOW",
    1: "MID",
    2: "HIGH"
}

yield_range_map = {
    0: "< 4 tons/hectare",
    1: "4 – 6 tons/hectare",
    2: "> 6 tons/hectare"
}


# 🔷 🔥 4. CHECK ALL CROPS

results = []

crop_probs = model_crop.predict_proba(base_input)[0]

for crop_num in range(5):

    input_data = base_input.copy()
    input_data['Crop'] = crop_num

    pred_yield = model_yield.predict(input_data)[0]

    score = pred_yield + crop_probs[crop_num]

    results.append((crop_num, pred_yield, score))


# 🔷 🔥 5. SELECT BEST CROP

best = max(results, key=lambda x: x[2])

best_crop = crop_map[best[0]]
best_yield_level = yield_level_map[best[1]]
best_yield_range = yield_range_map[best[1]]


# 🔷 🔥 6. FINAL OUTPUT (YOUR FORMAT)

print("\n--- FINAL RESULT ---")
print(f"Best Crop: {best_crop} ({best[0]})")
print(f"Yield Level: {best_yield_level} ({best[1]})")
print("Expected Yield:", best_yield_range)

Enter values (0=LOW, 1=MID, 2=HIGH)

Temperature: 0(<20°C), 1(20–30°C), 2(>30°C)
Temperature (0/1/2): 2

Rainfall: 0(<300 mm), 1(300–700 mm), 2(>700 mm)
Rainfall (0/1/2): 0

Humidity: 0(<50%), 1(50–75%), 2(>75%)
Humidity (0/1/2): 0

Weather: 0=Sunny, 1=Rainy, 2=Cloudy, 3=Stormy
Weather: 1

Soil: 0=Sandy, 1=Loamy, 2=Clay, 3=Peaty, 4=Silty
Soil: 4

--- FINAL RESULT ---
Best Crop: Rice (0)
Yield Level: LOW (0)
Expected Yield: < 4 tons/hectare
